The purpose of the project is to model the processes of a film rental payments by drawing from the Sakila dataset. In addition, this project uses interactions between customers, films, and stores and represents them in a fact table called payment. Each of the dimension tables are drawn from a different source of data (customers=csv, films=MongoDB, and stores=SQL database table). 

#This code import the necessary libraries. 

In [1]:
import os
import json
import numpy
import datetime
import certifi
import pandas as pd

import pymongo
import sqlalchemy
from sqlalchemy import create_engine, text

In [2]:
print(f"Running SQL Alchemy Version: {sqlalchemy.__version__}")
print(f"Running PyMongo Version: {pymongo.__version__}")

Running SQL Alchemy Version: 2.0.34
Running PyMongo Version: 4.16.0


#This code declares & assigns connection variables for the MongoDB Server, the MySQL Server & Databases. 

In [3]:
host_name = "localhost"
port = "3306"
user_id = "root"
pwd = "Marcel1n$"

mysql_args = {
    "uid": "root",
    "pwd": "Marcel1n$",
    "hostname": "localhost",
    "src_dbname": "sakila",
    "dbname": "sakila_dw"
}

mongodb_args = {
        "user_name": "jemarcelin14_db_user",
        "password": "6iWDKBwHzlNtj7Ok",   
        "cluster_name": "cluster1001",
        "cluster_subnet": "qnfuayc",
        "cluster_location": "atlas",
        "db_name": "sakila_dw"
}

#This code defines the functions for getting data from and setting data into databases

In [4]:
def get_sql_dataframe(sql_query, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    
    '''Invoke the pd.read_sql() function to query the database, and fill a Pandas DataFrame.'''
    dframe = pd.read_sql(text(sql_query), connection);
    connection.close()
    
    return dframe
    

def set_dataframe(df, table_name, pk_column, db_operation, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    db_connection = sqlEngine.connect()
    
    '''Invoke the Pandas DataFrame .to_sql( ) function to either create, or append to, a table'''
    if db_operation in ['insert', 'update']:
        if db_operation.lower() == "insert":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
            db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
                    
        elif db_operation.lower() == "update":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='append')

    else:
        print("The value supplied to the 'db_operation' parameter must be either 'insert' or 'update'.")
    
    db_connection.close()


def get_mongo_client(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.")
    
    else:
        if args["cluster_location"] == "atlas":
            connect_str = f"mongodb+srv://{args['user_name']}:{args['password']}@"
            connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
            client = pymongo.MongoClient(connect_str, tlsCAFile=certifi.where())
            
        elif args["cluster_location"] == "local":
            client = pymongo.MongoClient("mongodb://localhost:27017/")
        
    return client


def get_mongo_dataframe(mongo_client, db_name, collection, query):
    '''Query MongoDB, and fill a python list with documents to create a DataFrame'''
    db = mongo_client[db_name]
    dframe = pd.DataFrame(list(db[collection].find(query)))
    dframe.drop(['_id'], axis=1, inplace=True)
    
    return dframe


def set_mongo_collections(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()

In [5]:
from sqlalchemy import create_engine, text

conn_str = f"mysql+pymysql://{mysql_args['uid']}:{mysql_args['pwd']}@{mysql_args['hostname']}"

sqlEngine = create_engine(conn_str, pool_recycle=3600, isolation_level="AUTOCOMMIT")
connection = sqlEngine.connect()

dst_dbname = mysql_args["dbname"]

connection.execute(text(f"DROP DATABASE IF EXISTS `{dst_dbname}`;"))
connection.execute(text(f"CREATE DATABASE `{dst_dbname}`;"))
connection.execute(text(f"USE `{dst_dbname}`;"))

connection.close()

#This code populates MongoDB with Source Data. 

In [6]:
client = get_mongo_client(**mongodb_args)

data_dir = "/Users/jericmarcelin/Downloads"
json_files = {
    "films": "sakila_films.json"
}

set_mongo_collections(client, mongodb_args["db_name"], data_dir, json_files)

#This code extracts data from source MongoDB collections into DataFrames.

In [7]:
client = get_mongo_client(**mongodb_args)

query = {}
collection = "films"

df_films = get_mongo_dataframe(client,mongodb_args["db_name"],collection,query)
df_films.head(2)

,film_id,title,description,release_year,language_name,category_name,rental_duration,rental_rate,length,replacement_cost,rating,special_features
0,19,AMADEUS HOLY,A Emotional Display of a Pioneer And a Technic...,2006,English,Action,6,0.99,113,20.99,PG,"Commentaries,Deleted Scenes,Behind the Scenes"
1,21,AMERICAN CIRCUS,A Insightful Drama of a Girl And a Astronaut w...,2006,English,Action,3,4.99,129,17.99,R,"Commentaries,Behind the Scenes"


In [8]:
set_dataframe(df_films,"dim_films","film_key","insert",
    uid=mysql_args["uid"],
    pwd=mysql_args["pwd"],
    hostname=mysql_args["hostname"],
    dbname=mysql_args["dbname"]
)

This cleans up the data by dropping uneccesary columns. 

In [9]:
df_films.drop(columns=["description", "rating","category_name","special_features"], inplace=True)

In [10]:
dataframe = df_films
table_name = 'dim_films'
primary_key = 'film_key'
db_operation = "insert"

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args)

In [11]:
sql_film = "SELECT * FROM sakila_dw.dim_films;"
df_dim_films = get_sql_dataframe(sql_film, **mysql_args)
df_dim_films.head(2)

,film_key,film_id,title,release_year,language_name,rental_duration,rental_rate,length,replacement_cost
0,1,239,DOGMA FAMILY,2006,English,5,4.99,122,16.99
1,2,241,DONNIE ALLEY,2006,English,4,0.99,125,20.99


Import customers df from CSV.

In [13]:
df_customers = pd.read_csv("/Users/jericmarcelin/Downloads/sakila_customers.csv")
df_customers.head(2)

,customer_id,first_name,last_name,email,address,city,country,active
0,218,VERA,MCCOY,VERA.MCCOY@sakilacustomer.org,1168 Najafabad Parkway,Kabul,Afghanistan,1
1,441,MARIO,CHEATHAM,MARIO.CHEATHAM@sakilacustomer.org,1924 Shimonoseki Drive,Batna,Algeria,1


In [14]:
df_customers.drop(columns=["address"], inplace=True)

In [15]:
set_dataframe(df_customers,"dim_customer","customer_key","insert",
    uid=mysql_args["uid"],
    pwd=mysql_args["pwd"],
    hostname=mysql_args["hostname"],
    dbname=mysql_args["dbname"]
)

This code is used to upload from MySQL. 

In [17]:
sql_store = """
SELECT
    s.store_id,
    s.manager_staff_id,
    a.address,
    ci.city,
    co.country
FROM store s
JOIN address a
    ON s.address_id = a.address_id
JOIN city ci
    ON a.city_id = ci.city_id
JOIN country co
    ON ci.country_id = co.country_id;
"""

In [18]:
df_store = get_sql_dataframe(
    sql_store,
    uid=mysql_args["uid"],
    pwd=mysql_args["pwd"],
    hostname=mysql_args["hostname"],
    dbname=mysql_args["src_dbname"]
)

df_store.head(2)

,store_id,manager_staff_id,address,city,country
0,1,1,47 MySakila Drive,Lethbridge,Canada
1,2,2,28 MySQL Boulevard,Woodridge,Australia


This code is used to clean up the data frame before uploading. 

In [20]:
df_store.drop(columns=["manager_staff_id"], inplace=True)

In [21]:
set_dataframe(df_store,"dim_store","store_key","insert",
    uid=mysql_args["uid"],
    pwd=mysql_args["pwd"],
    hostname=mysql_args["hostname"],
    dbname=mysql_args["dbname"]
)

This code is used to upload dim_date.

In [ ]:
#Run SQL sakila date dimension code given from instructor. 
sql_dim_date = "SELECT date_key, full_date FROM sakila_dw.dim_date;"
df_dim_date = get_sql_dataframe(sql_dim_date, **mysql_args)
df_dim_date.full_date = df_dim_date.full_date.astype('datetime64[ns]').dt.date
df_dim_date.head(2)

,date_key,full_date
0,20050101,2005-01-01
1,20050102,2005-01-02


Check that DF are cleaned/transformed and are ready to be uploaded. 

In [26]:
sql_store_check = "SELECT * FROM sakila_dw.dim_store;"

df_dim_store = get_sql_dataframe(
    sql_store_check,
    uid=mysql_args["uid"],
    pwd=mysql_args["pwd"],
    hostname=mysql_args["hostname"],
    dbname=mysql_args["dbname"]
)

df_dim_store.head(2)

,store_key,store_id,address,city,country
0,1,1,47 MySakila Drive,Lethbridge,Canada
1,2,2,28 MySQL Boulevard,Woodridge,Australia


In [27]:
sql_customer_check = "SELECT * FROM sakila_dw.dim_customer;"

df_dim_customer = get_sql_dataframe(
    sql_customer_check,
    uid=mysql_args["uid"],
    pwd=mysql_args["pwd"],
    hostname=mysql_args["hostname"],
    dbname=mysql_args["dbname"]
)

df_dim_customer.head(2)

,customer_key,customer_id,first_name,last_name,email,city,country,active
0,1,218,VERA,MCCOY,VERA.MCCOY@sakilacustomer.org,Kabul,Afghanistan,1
1,2,300,JOHN,FARNSWORTH,JOHN.FARNSWORTH@sakilacustomer.org,Parbhani,India,1


In [28]:
sql_film_check = "SELECT * FROM sakila_dw.dim_films;"

df_dim_film = get_sql_dataframe(
    sql_film_check,
    uid=mysql_args["uid"],
    pwd=mysql_args["pwd"],
    hostname=mysql_args["hostname"],
    dbname=mysql_args["dbname"]
)

df_dim_film.head(2)

,film_key,film_id,title,release_year,language_name,rental_duration,rental_rate,length,replacement_cost
0,1,239,DOGMA FAMILY,2006,English,5,4.99,122,16.99
1,2,241,DONNIE ALLEY,2006,English,4,0.99,125,20.99


This takes the raw data from the sakila databse that reside in different tables and put in one dataset. 

In [31]:
sql_payments = """
SELECT
    p.payment_id,
    p.customer_id,
    i.film_id,
    i.store_id,
    p.payment_date,
    p.amount
FROM payment p
JOIN rental r
    ON p.rental_id = r.rental_id
JOIN inventory i
    ON r.inventory_id = i.inventory_id;
"""

df_payments = get_sql_dataframe(
    sql_payments,
    uid=mysql_args["uid"],
    pwd=mysql_args["pwd"],
    hostname=mysql_args["hostname"],
    dbname=mysql_args["src_dbname"]
)

Connects to database and pulls the payment data into pandas dataframe by storing into df_payments. 

In [32]:
df_payments = get_sql_dataframe(
    sql_payments,
    uid=mysql_args["uid"],
    pwd=mysql_args["pwd"],
    hostname=mysql_args["hostname"],
    dbname=mysql_args["src_dbname"]
)

df_payments.head(2)

,payment_id,customer_id,film_id,store_id,payment_date,amount
0,11630,431,1,1,2005-07-08 19:03:15,0.99
1,13956,518,1,1,2005-08-02 20:13:10,3.99


Converts the date into date_key that can be used later on in the project. 

In [33]:
df_dim_payment_date = df_dim_date.rename(columns={"full_date": "payment_date"})
df_payments["payment_date"] = df_payments["payment_date"].astype("datetime64[ns]").dt.date
df_payments = pd.merge(df_payments,df_dim_payment_date,on="payment_date",how="left")
df_payments.drop(["payment_date"], axis=1, inplace=True)
df_payments.head(2)

,payment_id,customer_id,film_id,store_id,amount,date_key
0,11630,431,1,1,0.99,20050708
1,13956,518,1,1,3.99,20050802


Converts the business keys into surrogate keys so that they can get called by the fact table. 

In [34]:
df_payments = pd.merge(df_payments,df_dim_customer[['customer_id', 'customer_key']],on='customer_id',how='left')
df_payments.drop(['customer_id'], axis=1, inplace=True)
df_payments.head(2)

,payment_id,film_id,store_id,amount,date_key,customer_key
0,11630,1,1,0.99,20050708,121
1,13956,1,1,3.99,20050802,371


In [35]:
df_payments = pd.merge(df_payments,df_dim_film[['film_id', 'film_key']],on='film_id',how='left')
df_payments.drop(['film_id'], axis=1, inplace=True)
df_payments.head(2)


,payment_id,store_id,amount,date_key,customer_key,film_key
0,11630,1,0.99,20050708,121,331
1,13956,1,3.99,20050802,371,331


In [36]:
df_payments = pd.merge(df_payments,df_dim_store[['store_id', 'store_key']],on='store_id',how='left')
df_payments.drop(['store_id'], axis=1, inplace=True)
df_payments.head(2)

,payment_id,amount,date_key,customer_key,film_key,store_key
0,11630,0.99,20050708,121,331,1
1,13956,3.99,20050802,371,331,1


Creates the fact table from the final dataframe, payments. The clean and transformed dataframe is uploaded into the data warehouse. 

In [37]:
set_dataframe(
    df_payments,
    "fact_payments",
    "fact_payments_key",
    "insert",
    uid=mysql_args["uid"],
    pwd=mysql_args["pwd"],
    hostname=mysql_args["hostname"],
    dbname=mysql_args["dbname"]
)

In [38]:
sql_fact_check = "SELECT * FROM sakila_dw.fact_payments;"

df_fact_check = get_sql_dataframe(sql_fact_check,
    uid=mysql_args["uid"],
    pwd=mysql_args["pwd"],
    hostname=mysql_args["hostname"],
    dbname=mysql_args["dbname"]
)

display(df_fact_check.head(5))

,fact_payments_key,payment_id,amount,date_key,customer_key,film_key,store_key
0,1,11630,0.99,20050708,121,331,1
1,2,10033,1.99,20050708,123,778,1
2,3,8123,2.99,20050728,254,886,1
3,4,12177,8.99,20050801,444,894,1
4,5,2014,1.99,20050820,207,886,1


This code shows that the warehouse allows for payment transactions to be analyzed given the infomation of films, stores, and customers. It joins the fact table with dim tables in order to calculate total revenue and payment counts.

In [ ]:
sql_sakila = """
SELECT
    f.length,
    s.country,
    SUM(fp.amount) AS total_revenue,
    COUNT(fp.payment_id) AS total_payments
FROM fact_payments fp
JOIN dim_films f
    ON fp.film_key = f.film_key
JOIN dim_store s
    ON fp.store_key = s.store_key
GROUP BY f.length, s.country
ORDER BY total_revenue DESC;
"""

display(get_sql_dataframe(sql_sakila, **mysql_args))

,length,country,total_revenue,total_payments
0,112,Canada,627.70,130
1,179,Australia,622.89,110
2,85,Australia,621.52,148
3,112,Australia,607.87,113
4,85,Canada,594.43,157
...,...,...,...,...
273,55,Australia,43.75,25
274,94,Australia,41.88,12
275,66,Canada,40.93,7
276,124,Canada,30.92,8
